# 06C (optional) PHA (GAFF2) + enzyme in water, OpenMM

**Workflow:** Optional enzyme–PHA alternative with OpenMM: 05A → 06C

**Before this notebook:** prepare a docked complex, see notebook 11. Run 05A for the PHA's GAFF2 files.

This notebook builds the enzyme + PHA complex in water with AmberTools `tleap` and prepares staged OpenMM MD, starting from the same docked complex PDB as 06B. For a PHA alone in water, use `06A_gaff2_gromacs_pha_in_water.ipynb`.

06B reproduces the project's research simulations. 06C is a different force-field setup, so results are not directly comparable with 06B.

| | 06B (research simulations) | 06C (this notebook's defaults) |
| --- | --- | --- |
| Protein | CHARMM36m | ff19SB |
| PHA | CGenFF (05B) | GAFF2 (05A; ABCG2 charges by default) |
| Water | CHARMM TIP3P | OPC |
| Ions / salt | SOD/CLA (NaCl), 0.05 M, neutralised | Na⁺/Cl⁻, 0.05 M, neutralised |
| Box | rectangular, 30 Å from the protein to the box edge (example: 11.4 nm cube) | rectangular, 3.0 nm padding around the complete complex |
| Engine | GROMACS | OpenMM |

The stage names, lengths, temperature and restraint strengths below follow 06B; the thermostat, barostat and non-bonded settings are OpenMM's and Amber's, so the MD protocol is similar, not identical.

| Stage (file prefix) | Settings |
| --- | --- |
| `step6.0_minimization` | tolerance 1000 kJ mol⁻¹ nm⁻¹, up to 5000 iterations, restraints |
| `step6.1_nvt` | 125 ps, 1 fs, 303.15 K (Langevin, 1 ps⁻¹), restraints |
| `step6.2_npt` | 500 ps, 2 fs, 303.15 K, 1 bar (Monte Carlo barostat), restraints |
| `step7_production` | 200 ns, 2 fs, NPT, frames every 100 ps, no restraints |
| restraints | 400 kJ mol⁻¹ nm⁻² on protein N/CA/C/O and polymer heavy atoms; 40 on other protein heavy atoms (as in CHARMM-GUI) |
| all stages | PME, 1.0 nm cut-off (Amber), HBonds constraints |

Every step that runs `tleap` or OpenMM is behind a `RUN_…` flag that defaults to `False`.
The final stage-specific intervals are NVT/NPT/production frames every **5/100/100 ps** and energy CSV every **1/2/2 ps**. OpenMM constraints use tolerance `1e-5` and PME error tolerance `5e-4`. See [the complete 06B–06C settings comparison](../docs/workflows/optional_openmm.md) for differences in minimisation, restraint references, nonbonded treatment and coupling algorithms.


## Caveat: GAFF2 with OPC

GAFF2 (and its ABCG2/AM1-BCC charges) was not parameterised against the OPC water model. GAFF2 + OPC (+ ff19SB for proteins) is a common combination, and ff19SB is recommended with OPC, but it has **not been validated for PHA**. Treat hydration-dependent results (solvation free energies, polymer–water interactions) with that in mind. The enzyme–PHA production runs used the CHARMM/GROMACS route (CHARMM36m + CGenFF + CHARMM TIP3P; notebook 06B), not this route.

## 1. Inputs (protein–polymer complex PDB + 05A polymer files)

- `COMPLEX_PDB` (**required**): the protein–polymer complex, e.g. the docking pose given to CHARMM-GUI. Protein hydrogens are not needed (tleap adds them).
- `LIGAND_RESNAME`: the polymer's residue name in that PDB (`LIG` for CHARMM-GUI inputs).
- `CHARMM_PROTEIN_ITP` (optional): CHARMM-GUI's `toppar/PROA.itp`, to give the histidines the same protonation states as the CHARMM/GROMACS system. Without it, histidines stay `HIS` (tleap uses HIE).
- `SYSTEM_NAME`, `MOL2_PATH`, `FRCMOD_PATH`, `TEMPLATE_SDF`: the 05A GAFF2 files for the **same oligomer** as in the complex (`<SYSTEM>.gaff2.mol2`, `.gaff2.frcmod`, `.antechamber.sdf`).
- `OUTPUT_DIR`: a **new** folder for the system; `INPUTS_DIR` receives the posed polymer and the protein PDB.
- Box and salt: `ENZYME_SOLVATION_SETTINGS` uses 3.0 nm padding (rectangular) and 0.05 M NaCl, matching 06B’s requested conditions. The box fits the complete complex rather than just the protein; water counts and final dimensions differ.
- Use a complete, repaired protein with deliberate protonation states, termini and disulfides; review these against 06B before simulation. Explicit HID/HIE/HIP and HSD/HSE/HSP states in the PDB are preserved. The optional ITP mapping assumes one protein chain with matching residue numbering.

In [ ]:
from pathlib import Path
import sys

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "iphasimulator").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))
from iphasimulator.amber_complex import pose_polymer_mol2, read_charmm_histidine_states, write_protein_pdb
from iphasimulator.amber_solvation import (
    ENZYME_POLYMER_IN_WATER_PROTOCOL,
    ENZYME_SOLVATION_SETTINGS,
    build_solvated_amber_system,
    run_openmm_short_test,
    run_openmm_stages,
    write_openmm_run_files,
    write_tleap_input,
)

COMPLEX_PDB = None         # required: protein–polymer complex PDB, e.g. Path(".../GK13_PHO4_pose2_complex.pdb")
LIGAND_RESNAME = "LIG"     # residue name of the polymer in COMPLEX_PDB
CHARMM_PROTEIN_ITP = None  # optional: CHARMM-GUI toppar/PROA.itp, to copy its histidine states (HSD -> HID, ...)
SYSTEM_NAME = "P3HB_4"     # the same oligomer as in the complex, parameterised in 05A
gaff2_dir = repo_root / "examples/output/md_tests" / SYSTEM_NAME / "gaff2"
MOL2_PATH = gaff2_dir / f"{SYSTEM_NAME}.gaff2.mol2"
FRCMOD_PATH = gaff2_dir / f"{SYSTEM_NAME}.gaff2.frcmod"
TEMPLATE_SDF = gaff2_dir / f"{SYSTEM_NAME}.antechamber.sdf"
OUTPUT_DIR = repo_root / "examples/output/md_tests" / SYSTEM_NAME / "openmm" / "enzyme_polymer_in_water"
INPUTS_DIR = OUTPUT_DIR.with_name(OUTPUT_DIR.name + "_inputs")  # posed polymer + protein PDB
JOB_NAME = f"{SYSTEM_NAME}_amber_enzyme_MD"

settings = ENZYME_SOLVATION_SETTINGS
protocol = ENZYME_POLYMER_IN_WATER_PROTOCOL
if COMPLEX_PDB is None:
    print("COMPLEX_PDB is required for this notebook: set it to the protein–polymer complex PDB.")
for label, path in (("mol2", MOL2_PATH), ("frcmod", FRCMOD_PATH), ("template sdf", TEMPLATE_SDF)):
    print(f"{label + ' found:':<20}{path.is_file()} {path}")
print("output:", OUTPUT_DIR, "(exists — choose a new folder)" if OUTPUT_DIR.exists() else "")

## 2. Build complex in water (tleap: ff19SB + GAFF2 + OPC + ions)

First the complex is split (RDKit, no tleap): the protein becomes `protein.pdb` (ligand and hydrogens removed, histidine states applied), and the 05A GAFF2 polymer is moved into the docked pose (heavy atoms matched by connectivity, hydrogens placed, atom types and charges unchanged); its stereocentres are printed and should all be R.

Then `tleap` sources ff19SB, GAFF2 and OPC, combines protein and posed polymer, solvates (3.0 nm padding, rectangular), neutralises and adds 0.05 M Na⁺/Cl⁻, and saves `system.prmtop`/`system.inpcrd`.

Writing the run files puts `run_openmm_md.py` (stand-alone; needs only OpenMM), `protocol.json`, `run_step6_local.sh` and `run_hpc_equilibration_production.slurm` next to `system.prmtop`, with the same stage names as the GROMACS folders: step 6.0 locally, then 6.1, 6.2 and 7 on the cluster. Enable `RUN_PREPARE_INPUTS` first, then `RUN_TLEAP`; both refuse existing output directories. `PHA` must be the residue name in the 05A MOL2 (or change `polymer_residues` to its actual name when writing run files).

In [ ]:
RUN_PREPARE_INPUTS = False

if not RUN_PREPARE_INPUTS:
    print("Set RUN_PREPARE_INPUTS = True after checking your complex and 05A files.")
elif COMPLEX_PDB is None:
    print("Set COMPLEX_PDB in step 1 to prepare the protein and posed polymer.")
else:
    for required in (Path(COMPLEX_PDB), MOL2_PATH, FRCMOD_PATH, TEMPLATE_SDF):
        if not required.is_file():
            raise FileNotFoundError(f"Input not found: {required}")
    INPUTS_DIR.mkdir(parents=True, exist_ok=False)
    histidines = read_charmm_histidine_states(CHARMM_PROTEIN_ITP) if CHARMM_PROTEIN_ITP else None
    protein = write_protein_pdb(COMPLEX_PDB, INPUTS_DIR / "protein.pdb",
                                ligand_resname=LIGAND_RESNAME, histidine_states=histidines)
    pose = pose_polymer_mol2(MOL2_PATH, TEMPLATE_SDF, COMPLEX_PDB, INPUTS_DIR / f"{SYSTEM_NAME}.posed.mol2",
                             ligand_resname=LIGAND_RESNAME)
    print(f"Protein: {protein.residues} residues; histidines {sorted(set(protein.histidines.values())) or 'none'}")
    print(f"Polymer: {pose.heavy_atoms} heavy atoms on the docked pose, {pose.hydrogens_placed} hydrogens placed")
    print("Stereocentres:", ", ".join(f"{atom} {label}" for atom, label in pose.stereocentres))

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as preview_dir:
    print(write_tleap_input(Path(preview_dir) / "tleap.in", mol2_name=f"{SYSTEM_NAME}.posed.mol2",
                            frcmod_name=FRCMOD_PATH.name, protein_pdb_name="protein.pdb",
                            settings=settings, salt_pairs=0))

RUN_TLEAP = False

if not RUN_TLEAP:
    print("Set RUN_TLEAP = True to build the complex in water with tleap.")
else:
    assert COMPLEX_PDB is not None, "COMPLEX_PDB is required: set it in step 1 to the protein–polymer complex PDB."
    system = build_solvated_amber_system(
        INPUTS_DIR / f"{SYSTEM_NAME}.posed.mol2", FRCMOD_PATH, OUTPUT_DIR,
        protein_pdb_path=INPUTS_DIR / "protein.pdb", settings=settings,
    )
    print("Waters added:        ", system.water_count)
    print("Salt ion pairs added:", system.salt_ion_pairs)
    print("Ion parameter files: ", ", ".join(system.ion_parameter_files) or "none reported")
    for path in write_openmm_run_files(OUTPUT_DIR, protocol, job_name=JOB_NAME, polymer_residues=("PHA",)):
        print("Wrote", path)

## 3. OpenMM short test

Minimisation (with the restraints), then 10 ps NVT and 10 ps NPT with the protocol's temperature, pressure, time steps and restraints, written to `OUTPUT_DIR/short_test/`. It reports the potential energy and density after each stage. Ten picoseconds only show that the system is stable, not that it is equilibrated.

In [ ]:
RUN_OPENMM_TEST = False

if not RUN_OPENMM_TEST:
    print("Set RUN_OPENMM_TEST = True to run the short OpenMM test.")
else:
    for result in run_openmm_short_test(OUTPUT_DIR, platform_name=None):  # e.g. "CUDA"
        print(f"{result['stage']}: Epot {result['potential_energy_kj_mol']:.1f} kJ/mol, density {result['density_g_ml']:.3f} g/mL")

## 4. Production files for 07

The run files were written in step 2. Run step 6.0 (restrained minimisation) locally before handing the folder to `07_hpc_execution.ipynb` (OpenMM section), which explains submission of `run_hpc_equilibration_production.slurm` for steps 6.1, 6.2 and 7. Every MD stage writes its own `.chk` and `.restart.json`; resubmission resumes incomplete stages and takes no further steps in completed stages. Keep inputs and `protocol.json` unchanged. OpenMM checkpoints require compatible hardware, platform and OpenMM version. Existing run folders retain their copied runner; regenerate into a new folder to use these fixes. The short test writes a separate `short_test/` state and does not initialise the full run.

In [ ]:
RUN_LOCAL_MINIMIZATION = False

if not RUN_LOCAL_MINIMIZATION:
    print("Set RUN_LOCAL_MINIMIZATION = True to run step 6.0 locally.")
else:
    print(run_openmm_stages(OUTPUT_DIR, ["step6.0_minimization"]))

for name in ("system.prmtop", "system.inpcrd", "run_openmm_md.py", "protocol.json", "step6.0_minimization.xml", "run_hpc_equilibration_production.slurm"):
    print(f"{'found  ' if (OUTPUT_DIR / name).exists() else 'missing'}  {name}")
print(f"On the cluster: cd {OUTPUT_DIR} && sbatch run_hpc_equilibration_production.slurm")